# Example-01: Estimation of parameters (harmonic sum)

In [1]:
# Import

import numpy
import torch
import yaml

import sys
sys.path.append('..')

from harmonica.util import mod
from harmonica.window import Window
from harmonica.data import Data
from harmonica.frequency import Frequency
from harmonica.filter import Filter
from harmonica.decomposition import Decomposition

import matplotlib.pyplot as plt

torch.set_printoptions(precision=12, sci_mode=True)
print(torch.cuda.is_available())

True


In [2]:
# Set data type and device

dtype = torch.float64
device = torch.device('cpu')

In [3]:
# Estimate parameters and standard errors (with random frequency)

# Set parameters

length = 1024

# Set window

w = Window(length, 'cosine_window', 1.0, dtype=dtype, device=device)

# Set noise parameters

s_x = torch.tensor([0.05, 0.01], dtype=dtype, device=device)
s_f = 1.0E-5

# Set data

t = torch.linspace(0, length - 1, length, dtype=dtype, device=device)
x = torch.stack([0.25*torch.cos(2.0*numpy.pi*0.12*t + 0.1), 0.75*torch.cos(2.0*numpy.pi*0.12*t + 0.5)])

# Estimate parameters and errors (direct)

p1, s1 = Decomposition.harmonic_sum(0.12, w.window, x, error=True, sigma=s_x, sigma_frequency=s_f)

# Estimate parameters and errors (automatic)

p2, s2 = Decomposition.harmonic_sum_automatic(0.12, w.window, x, error=True, sigma=s_x, sigma_frequency=s_f)

# Estimate errors (average over noise realizations)

box = []
for _ in range(8192):
    y = x + s_x.reshape(-1, 1)*torch.randn_like(x, dtype=dtype, device=device)
    out, _ = Decomposition.harmonic_sum(0.12 + 1.0E-5*torch.randn(1, dtype=dtype, device=device), w.window, y)
    box.append(out)    
box = torch.stack(box).std(-3)

# Compare parameters

print(p1)
print(p2)
print()

# Compare errors

print(s1)
print(s2)
print(box)
print()

tensor([[2.487510441567e-01, -2.495835633644e-02, 2.500000030401e-01, 1.000000075225e-01],
        [6.581869317981e-01, -3.595691566478e-01, 7.500000104014e-01, 4.999999965175e-01]],
       dtype=torch.float64)
tensor([[2.487510441567e-01, -2.495835633644e-02, 2.500000030401e-01, 1.000000075225e-01],
        [6.581869317981e-01, -3.595691566478e-01, 7.500000104014e-01, 4.999999965175e-01]],
       dtype=torch.float64)

tensor([[2.822920460161e-03, 8.447543997630e-03, 2.706329386825e-03, 3.394245931339e-02],
        [1.157996334008e-02, 2.118073069991e-02, 5.412658773651e-04, 3.217800267359e-02]],
       dtype=torch.float64)
tensor([[2.822920460161e-03, 8.447543997630e-03, 2.706329386825e-03, 3.394245931339e-02],
        [1.157996334008e-02, 2.118073069991e-02, 5.412658773651e-04, 3.217800267359e-02]],
       dtype=torch.float64)
tensor([[2.802782711160e-03, 8.356512643284e-03, 2.674990280570e-03, 3.360273931306e-02],
        [1.148432343642e-02, 2.099971340980e-02, 5.490713943031e-04, 

In [4]:
# Estimate amplitude and phase from cos & sin values
# Note, for error propagation cos & sin are treated as independent random variables (which is not true)

c, s, *_ = p1.T
sigma_c, sigma_s, *_ = s1.T

value, error = Decomposition.amplitude(c, s, sigma_c=sigma_c, sigma_s=sigma_s)
print(value)
print(error)

value, error = Decomposition.phase(c, s, sigma_c=sigma_c, sigma_s=sigma_s)
print(value)
print(error)

tensor([2.500000030401e-01, 7.500000104014e-01], dtype=torch.float64)
tensor([2.932693465157e-03, 1.436625917563e-02], dtype=torch.float64)
tensor([1.000000075225e-01, 4.999999965175e-01], dtype=torch.float64)
tensor([3.364025846168e-02, 2.586561821930e-02], dtype=torch.float64)


In [5]:
# Estimate cos & sin amplitudes for knowns frequencies using OLS

basis = torch.tensor([0.12], dtype=dtype, device=device)
for signal in x:
    out, _ = Decomposition.fit_ols(signal, basis)
    print(out)

tensor([[2.487510413195e-01, -2.495835416171e-02]], dtype=torch.float64)
tensor([[6.581869214178e-01, -3.595691539531e-01]], dtype=torch.float64)


In [7]:
# Estimate cos & sin amplitudes for a given frequency dictionary using OLS and OMP

signal  = 0.50*torch.cos(2.0*numpy.pi*1*0.12*t) + 0.10*torch.sin(2.0*numpy.pi*1*0.12*t)
signal += 0.10*torch.cos(2.0*numpy.pi*3*0.12*t) + 0.05*torch.sin(2.0*numpy.pi*3*0.12*t)
signal += 0.01*torch.rand_like(signal)

matrix = Filter.make_matrix(signal.unsqueeze(0))
_, sigma = Filter.svd_optimal(matrix[:, :32])
sigma = sigma.item()

basis = torch.tensor([*Frequency.harmonics(10, [0.12]).values()], dtype=dtype, device=device)

out, _ = Decomposition.fit_ols(signal, basis, length=256)
print(out)
print()

out, _ = Decomposition.fit_omp(signal, basis, tol=3.0*sigma, length=256)
print(out)
print()

tensor([[ 5.001098124591e-01,  9.996363148893e-02],
        [-1.523856660461e-04,  2.912083559217e-04],
        [ 1.003110086310e-01,  4.989457119988e-02],
        [-5.107793392398e-04, -2.870064917945e-04],
        [-2.640687218915e-04, -2.915746286255e-04],
        [-3.330001018499e-04,  1.699976693743e-04],
        [-4.972470154018e-05,  3.717176652639e-04],
        [-1.328429993075e-04,  4.317467987948e-04],
        [ 3.044452875112e-04, -4.601302592127e-05],
        [ 6.158678239156e-04,  2.483680001817e-04]], dtype=torch.float64)

tensor([[5.001452935791e-01, 9.988767178799e-02],
        [0.000000000000e+00, 0.000000000000e+00],
        [1.002782503251e-01, 4.990379997897e-02],
        [0.000000000000e+00, 0.000000000000e+00],
        [0.000000000000e+00, 0.000000000000e+00],
        [0.000000000000e+00, 0.000000000000e+00],
        [0.000000000000e+00, 0.000000000000e+00],
        [0.000000000000e+00, 0.000000000000e+00],
        [0.000000000000e+00, 0.000000000000e+00],
       